# RAG-Based E-commerce Customer Support Chatbot

**One Kaggle notebook — all models and the integrated application**  


This notebook implements the complete project in one reproducible workflow:

1. Language detection using character TF-IDF and LinearSVC.
2. Sentiment/tone classification using a fine-tuned DistilBERT transformer.
3. Intent routing using word/character TF-IDF and LinearSVC.
4. RAG using Sentence Transformers, FAISS, the Bitext knowledge base, and Groq `openai/gpt-oss-20b`.
5. End-to-end routing, multilingual replies, priority handling, evaluation, artifact export, and local-deployment bundle creation.

Run the notebook **from top to bottom**. Explanations are included before each important code cell.

## Kaggle setup

Before running anything:

1. Import this `.ipynb` into a new Kaggle notebook.
2. Open **Notebook options**, enable **Internet**, and select a **GPU accelerator**.
3. Create a Groq API key.
4. In Kaggle, open **Add-ons → Secrets**, add a secret named exactly `GROQ_API_KEY`, and enable it for the notebook.
5. Select **Run All**. The notebook saves intermediate checkpoints after every model, so if a later cell fails you can resume from the completed sections.
6. At the end, save a Kaggle version and download `complete_rag_artifacts.zip`.

Do not put the Groq key directly in any code or Markdown cell.

## Configuration

`QUICK_MODE=False` is the recommended final-project setting. It uses the complete language and intent datasets and trains sentiment for two epochs. Set it to `True` only for an initial pipeline test; results from quick mode should not be reported as final metrics.

In [1]:
%pip install -q pandas pyarrow scikit-learn joblib transformers sentence-transformers faiss-cpu groq safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 9.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os, re, json, random, time, gc, shutil
import numpy as np
import pandas as pd
import torch
import joblib
import faiss
from torch.utils.data import Dataset, DataLoader
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sentence_transformers import SentenceTransformer
from groq import Groq

SEED = 42
QUICK_MODE = False
SENTIMENT_EPOCHS = 1 if QUICK_MODE else 2
SENTIMENT_BATCH_SIZE = 32
MAX_LENGTH = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
ARTIFACTS = WORK/'artifacts'
for folder in ['language','sentiment','intent','rag']:
    (ARTIFACTS/folder).mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print({'device':str(DEVICE), 'quick_mode':QUICK_MODE, 'artifacts':str(ARTIFACTS)})
if DEVICE.type != 'cuda':
    print('WARNING: Select a Kaggle GPU before training DistilBERT.')

{'device': 'cuda', 'quick_mode': False, 'artifacts': '/kaggle/working/artifacts'}


## Download datasets

The notebook downloads the official CSV/Parquet files directly from Hugging Face. This keeps the schema visible and avoids dependency problems caused by dataset-loader changes.

- Language data: `papluca/language-identification` with official train, validation, and test splits.
- Sentiment data: `dair-ai/emotion` with six original emotions.
- Intent and RAG data: Bitext customer-support instructions, responses, intents, and categories.

In [3]:
LANG_BASE = 'https://huggingface.co/datasets/papluca/language-identification/resolve/main'
lang_train = pd.read_csv(f'{LANG_BASE}/train.csv')
lang_valid = pd.read_csv(f'{LANG_BASE}/valid.csv')
lang_test = pd.read_csv(f'{LANG_BASE}/test.csv')

EMOTION_BASE = 'https://huggingface.co/datasets/dair-ai/emotion/resolve/main/split'
emotion_train = pd.read_parquet(f'{EMOTION_BASE}/train-00000-of-00001.parquet')
emotion_valid = pd.read_parquet(f'{EMOTION_BASE}/validation-00000-of-00001.parquet')
emotion_test = pd.read_parquet(f'{EMOTION_BASE}/test-00000-of-00001.parquet')

BITEXT_URL = 'https://huggingface.co/datasets/bitext/Bitext-customer-support-llm-chatbot-training-dataset/resolve/main/Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv'
bitext = pd.read_csv(BITEXT_URL)

assert {'text','labels'} <= set(lang_train.columns)
assert {'text','label'} <= set(emotion_train.columns)
assert {'instruction','response','category','intent'} <= set(bitext.columns)
print('Language:', lang_train.shape, lang_valid.shape, lang_test.shape)
print('Emotion:', emotion_train.shape, emotion_valid.shape, emotion_test.shape)
print('Bitext:', bitext.shape)
print('Bitext intents:', bitext['intent'].nunique())

Language: (70000, 2) (10000, 2) (10000, 2)
Emotion: (16000, 2) (2000, 2) (2000, 2)
Bitext: (26872, 5)
Bitext intents: 27


# Module 1 — Language detection

Character n-grams are appropriate for language identification because scripts, accents, and recurring character sequences remain informative even in short or misspelled text. The complete preprocessing and classifier are stored in a single scikit-learn pipeline to prevent training/inference mismatch.

In [4]:
def normalize_text(text):
    return re.sub(r'\s+', ' ', str(text).replace('\n',' ').replace('\t',' ')).strip()

for frame in (lang_train, lang_valid, lang_test):
    frame['text'] = frame['text'].map(normalize_text)
    frame.dropna(subset=['labels'], inplace=True)
    frame.drop_duplicates(subset=['text','labels'], inplace=True)

if QUICK_MODE:
    lang_fit = lang_train.groupby('labels', group_keys=False).head(1000).copy()
else:
    lang_fit = lang_train.copy()

language_model = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char', ngram_range=(2,5), min_df=2, max_features=250_000,
        sublinear_tf=True, lowercase=True, dtype=np.float32
    )),
    ('classifier', LinearSVC(C=2.0, class_weight='balanced', random_state=SEED))
])
language_model.fit(lang_fit['text'], lang_fit['labels'])
lang_valid_pred = language_model.predict(lang_valid['text'])
lang_test_pred = language_model.predict(lang_test['text'])
print('Validation accuracy:', round(accuracy_score(lang_valid['labels'],lang_valid_pred),4))
print('Test accuracy:', round(accuracy_score(lang_test['labels'],lang_test_pred),4))
print(classification_report(lang_test['labels'],lang_test_pred,digits=4))

Validation accuracy: 0.9954
Test accuracy: 0.995
              precision    recall  f1-score   support

          ar     1.0000    0.9977    0.9989       440
          bg     0.9977    1.0000    0.9989       440
          de     1.0000    1.0000    1.0000       500
          el     1.0000    1.0000    1.0000       440
          en     0.9960    1.0000    0.9980       500
          es     0.9980    0.9980    0.9980       500
          fr     0.9980    1.0000    0.9990       500
          hi     1.0000    0.9636    0.9815       440
          it     0.9938    0.9959    0.9948       484
          ja     1.0000    1.0000    1.0000       500
          nl     0.9959    0.9979    0.9969       484
          pl     1.0000    0.9938    0.9969       481
          pt     0.9938    0.9979    0.9959       484
          ru     1.0000    0.9977    0.9989       440
          sw     0.9339    0.9955    0.9637       440
          th     1.0000    0.9977    0.9989       440
          tr     0.9932    1.000

In [5]:
lang_errors = lang_test.loc[lang_test_pred != lang_test['labels'], ['text','labels']].copy()
lang_errors['predicted'] = lang_test_pred[lang_test_pred != lang_test['labels']]
display(lang_errors.head(10))

joblib.dump(language_model, ARTIFACTS/'language'/'language_pipeline.joblib', compress=3)
lang_meta = {
    'model':'character TF-IDF + LinearSVC',
    'dataset':'papluca/language-identification',
    'languages':sorted(lang_train['labels'].unique().tolist()),
    'validation_accuracy':float(accuracy_score(lang_valid['labels'],lang_valid_pred)),
    'test_accuracy':float(accuracy_score(lang_test['labels'],lang_test_pred)),
    'quick_mode':QUICK_MODE, 'seed':SEED
}
(ARTIFACTS/'language'/'metadata.json').write_text(json.dumps(lang_meta,indent=2),encoding='utf-8')
assert joblib.load(ARTIFACTS/'language'/'language_pipeline.joblib').predict(['Where is my order?'])[0] == 'en'
print('LANGUAGE MODULE PASSED')

,text,labels,predicted
501,aadhunik mahilaen Romantic shira mein maheen h...,hi,sw
718,"Us din mai apna wheel ghooma raha tha, mjhe na...",ur,sw
1675,is nivesh ne 60 gharo ka navikaran aur maamool...,hi,sw
1910,To także kwestia gustu.,pl,sw
2036,Apple iPad mini debiutuje,pl,sw
2145,"To mai gaya, mai Washington D.C. gaya aur mai ...",ur,sw
2500,คนที่ L'academie Internationale des Arts et de...,th,fr
2842,"gupt karya ke liye, beshak, White House Counte...",hi,sw
2877,ek graahak kee anupasthiti ke dauraan nirantar...,hi,nl
3192,Romney Wins Nevada Caucus,pl,pt


LANGUAGE MODULE PASSED


# Module 2 — Sentiment and tone

The source dataset labels sadness, joy, love, anger, fear, and surprise. For customer-support routing, the notebook explicitly maps them to:

- Negative: sadness, anger, fear.
- Neutral: surprise.
- Positive: joy, love.

This mapping is an engineering simplification: surprise is not always neutral, and Twitter text differs from customer-support language. The report must state both limitations. Class-weighted loss reduces the effect of the small neutral bucket.

In [6]:
emotion_to_bucket = {0:0, 1:2, 2:2, 3:0, 4:0, 5:1}
id2label = {0:'negative', 1:'neutral', 2:'positive'}
label2id = {v:k for k,v in id2label.items()}

for frame in (emotion_train,emotion_valid,emotion_test):
    frame['text'] = frame['text'].map(normalize_text)
    frame['bucket'] = frame['label'].map(emotion_to_bucket).astype(int)

if QUICK_MODE:
    sentiment_fit = emotion_train.groupby('bucket',group_keys=False).head(1500).sample(frac=1,random_state=SEED)
else:
    sentiment_fit = emotion_train.copy()
print(sentiment_fit['bucket'].map(id2label).value_counts())

SENTIMENT_BASE_MODEL='distilbert-base-uncased'
sentiment_tokenizer=AutoTokenizer.from_pretrained(SENTIMENT_BASE_MODEL)

class ToneDataset(Dataset):
    def __init__(self, frame):
        self.texts=frame['text'].tolist(); self.labels=frame['bucket'].tolist()
    def __len__(self): return len(self.texts)
    def __getitem__(self,idx):
        enc=sentiment_tokenizer(self.texts[idx],truncation=True,max_length=MAX_LENGTH,
                                padding='max_length',return_tensors='pt')
        item={k:v.squeeze(0) for k,v in enc.items()}
        item['labels']=torch.tensor(self.labels[idx],dtype=torch.long)
        return item

train_loader=DataLoader(ToneDataset(sentiment_fit),batch_size=SENTIMENT_BATCH_SIZE,
                        shuffle=True,num_workers=2,pin_memory=True)
valid_loader=DataLoader(ToneDataset(emotion_valid),batch_size=SENTIMENT_BATCH_SIZE*2,
                        shuffle=False,num_workers=2,pin_memory=True)
test_loader=DataLoader(ToneDataset(emotion_test),batch_size=SENTIMENT_BATCH_SIZE*2,
                       shuffle=False,num_workers=2,pin_memory=True)

bucket
negative    8762
positive    6666
neutral      572
Name: count, dtype: int64


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
sentiment_model=AutoModelForSequenceClassification.from_pretrained(
    SENTIMENT_BASE_MODEL,num_labels=3,id2label=id2label,label2id=label2id
).to(DEVICE)
counts=sentiment_fit['bucket'].value_counts().sort_index().values
weights=len(sentiment_fit)/(3*counts)
loss_fn=torch.nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float,device=DEVICE))
optimizer=torch.optim.AdamW(sentiment_model.parameters(),lr=2e-5,weight_decay=0.01)
total_steps=len(train_loader)*SENTIMENT_EPOCHS
scheduler=get_linear_schedule_with_warmup(optimizer,int(0.1*total_steps),total_steps)

@torch.no_grad()
def evaluate_tone(loader, model=None):
    model=model or sentiment_model
    model.eval(); losses=[]; gold=[]; pred=[]
    for batch in loader:
        labels=batch.pop('labels').to(DEVICE)
        inputs={k:v.to(DEVICE) for k,v in batch.items()}
        logits=model(**inputs).logits
        losses.append(loss_fn(logits,labels).item())
        gold.extend(labels.cpu().tolist()); pred.extend(logits.argmax(1).cpu().tolist())
    return float(np.mean(losses)),accuracy_score(gold,pred),f1_score(gold,pred,average='macro'),gold,pred

best_f1=-1
for epoch in range(SENTIMENT_EPOCHS):
    sentiment_model.train(); running=[]
    for step,batch in enumerate(train_loader,1):
        labels=batch.pop('labels').to(DEVICE)
        inputs={k:v.to(DEVICE) for k,v in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        logits=sentiment_model(**inputs).logits
        loss=loss_fn(logits,labels); loss.backward()
        torch.nn.utils.clip_grad_norm_(sentiment_model.parameters(),1.0)
        optimizer.step(); scheduler.step(); running.append(loss.item())
        if step%200==0:
            print(f'Epoch {epoch+1} step {step}/{len(train_loader)} loss={np.mean(running[-200:]):.4f}')
    val_loss,val_acc,val_f1,_,_=evaluate_tone(valid_loader)
    print(f'Epoch {epoch+1}: train_loss={np.mean(running):.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f} val_macro_f1={val_f1:.4f}')
    if val_f1>best_f1:
        best_f1=val_f1
        sentiment_model.save_pretrained(ARTIFACTS/'sentiment',safe_serialization=True)
        sentiment_tokenizer.save_pretrained(ARTIFACTS/'sentiment')
        print('Saved best sentiment checkpoint')

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1 step 200/500 loss=0.8209
Epoch 1 step 400/500 loss=0.1845
Epoch 1: train_loss=0.4274 val_loss=0.1149 val_acc=0.9695 val_macro_f1=0.9342


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best sentiment checkpoint
Epoch 2 step 200/500 loss=0.0927
Epoch 2 step 400/500 loss=0.1165
Epoch 2: train_loss=0.1005 val_loss=0.1084 val_acc=0.9725 val_macro_f1=0.9358


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best sentiment checkpoint


In [8]:
sentiment_model=AutoModelForSequenceClassification.from_pretrained(ARTIFACTS/'sentiment').to(DEVICE)
test_loss,tone_test_acc,tone_test_f1,tone_gold,tone_pred=evaluate_tone(test_loader,sentiment_model)
print(f'Test accuracy={tone_test_acc:.4f}; macro-F1={tone_test_f1:.4f}')
print(classification_report(tone_gold,tone_pred,target_names=[id2label[i] for i in range(3)],digits=4))

sentiment_meta={
    'base_model':SENTIMENT_BASE_MODEL,'dataset':'dair-ai/emotion',
    'mapping':{'negative':['sadness','anger','fear'],'neutral':['surprise'],'positive':['joy','love']},
    'test_accuracy':float(tone_test_acc),'test_macro_f1':float(tone_test_f1),
    'epochs':SENTIMENT_EPOCHS,'quick_mode':QUICK_MODE,'seed':SEED,
    'limitations':['Twitter-to-support domain shift','Surprise is an imperfect neutral proxy']
}
(ARTIFACTS/'sentiment'/'training_metadata.json').write_text(json.dumps(sentiment_meta,indent=2),encoding='utf-8')

@torch.no_grad()
def predict_sentiment(text):
    batch=sentiment_tokenizer(text,return_tensors='pt',truncation=True,max_length=MAX_LENGTH).to(DEVICE)
    probs=torch.softmax(sentiment_model(**batch).logits,1)[0].cpu().numpy()
    idx=int(probs.argmax())
    return id2label[idx],float(probs[idx])

for text in ['I am furious about my refund!','Can you explain delivery time?','Thank you for the excellent help!']:
    print(text,'->',predict_sentiment(text))
print('SENTIMENT MODULE PASSED')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Test accuracy=0.9720; macro-F1=0.9126
              precision    recall  f1-score   support

    negative     0.9831    0.9704    0.9767      1080
     neutral     0.6703    0.9242    0.7771        66
    positive     0.9905    0.9778    0.9841       854

    accuracy                         0.9720      2000
   macro avg     0.8813    0.9575    0.9126      2000
weighted avg     0.9760    0.9720    0.9733      2000

I am furious about my refund! -> ('negative', 0.9961198568344116)
Can you explain delivery time? -> ('negative', 0.7403881549835205)
Thank you for the excellent help! -> ('positive', 0.9960544109344482)
SENTIMENT MODULE PASSED


# Module 3 — Intent routing

Bitext provides 27 gold intent labels. The dictionary below condenses all of them into the seven project routes. Address and cancellation-fee requests are treated as order management; contact-agent requests are routed as complaints/escalations.

Bitext lacks explicit small-talk and out-of-scope labels, so the notebook adds a small, transparent educational set for those two routes. Replace it with collected real examples in a production system.

In [9]:
bitext=bitext.dropna(subset=['instruction','response','intent','category']).copy()
bitext['instruction']=bitext['instruction'].map(normalize_text)
bitext['response']=bitext['response'].map(normalize_text)
bitext.drop_duplicates(subset=['instruction','intent'],inplace=True)

ROUTE_MAP={
 'track_order':'order_status','delivery_options':'order_status','delivery_period':'order_status',
 'cancel_order':'order_management','change_order':'order_management','place_order':'order_management',
 'check_cancellation_fee':'order_management','change_shipping_address':'order_management',
 'set_up_shipping_address':'order_management',
 'check_invoice':'billing_and_refunds','get_invoice':'billing_and_refunds',
 'get_refund':'billing_and_refunds','check_refund_policy':'billing_and_refunds',
 'track_refund':'billing_and_refunds','payment_issue':'billing_and_refunds',
 'check_payment_methods':'billing_and_refunds',
 'create_account':'account_management','delete_account':'account_management',
 'edit_account':'account_management','switch_account':'account_management',
 'recover_password':'account_management','registration_problems':'account_management',
 'newsletter_subscription':'account_management',
 'complaint':'complaint','review':'complaint','contact_customer_service':'complaint',
 'contact_human_agent':'complaint'
}
source_intents=set(bitext['intent'].unique())
assert source_intents==set(ROUTE_MAP), f'Unmapped={source_intents-set(ROUTE_MAP)}; extra={set(ROUTE_MAP)-source_intents}'
bitext['route_intent']=bitext['intent'].map(ROUTE_MAP)

greetings=['hello','hi','hey','good morning','good afternoon','good evening','how are you',
 'nice to meet you','thanks','thank you','many thanks','thanks for your help','I appreciate your help',
 'goodbye','bye','see you later','have a nice day','talk to you later','cheers']
out_scope=['what is the weather today','tell me a joke','who is the president','write Python code for me',
 'recommend a movie','what is the capital of France','solve my mathematics homework','play music',
 'book a hotel room','give me medical advice','how do I cook pasta','what is the football score',
 'translate this poem','create an image','explain quantum physics','find me a job','what time is it']
rows=[]
for text in greetings:
    for suffix in ['',' please','!',' there']:
        rows.append({'instruction':text+suffix,'route_intent':'greeting_goodbye_gratitude'})
for text in out_scope:
    for prefix in ['','please ','can you ','I want you to ']:
        rows.append({'instruction':prefix+text,'route_intent':'out_of_scope'})
augmented=pd.DataFrame(rows).drop_duplicates()
intent_df=pd.concat([bitext[['instruction','route_intent']],augmented],ignore_index=True)
if QUICK_MODE:
    intent_df=intent_df.groupby('route_intent',group_keys=False).head(1500).copy()
print(intent_df['route_intent'].value_counts())

route_intent
billing_and_refunds           6435
account_management            6412
order_management              5263
complaint                     3988
order_status                  2456
greeting_goodbye_gratitude      76
out_of_scope                    68
Name: count, dtype: int64


In [10]:
X_train,X_test,y_train,y_test=train_test_split(
    intent_df['instruction'],intent_df['route_intent'],test_size=0.20,
    random_state=SEED,stratify=intent_df['route_intent']
)
intent_features=FeatureUnion([
    ('word',TfidfVectorizer(ngram_range=(1,2),min_df=2,max_features=100_000,sublinear_tf=True)),
    ('char',TfidfVectorizer(analyzer='char_wb',ngram_range=(3,5),min_df=2,max_features=120_000,sublinear_tf=True))
])
intent_model=Pipeline([
    ('features',intent_features),
    ('classifier',LinearSVC(C=2.0,class_weight='balanced',random_state=SEED))
])
intent_model.fit(X_train,y_train)
intent_pred=intent_model.predict(X_test)
intent_acc=accuracy_score(y_test,intent_pred)
intent_f1=f1_score(y_test,intent_pred,average='macro')
print('Accuracy:',round(intent_acc,4),'Macro-F1:',round(intent_f1,4))
print(classification_report(y_test,intent_pred,digits=4))

intent_errors=pd.DataFrame({'text':X_test,'gold':y_test,'predicted':intent_pred})
display(intent_errors[intent_errors.gold!=intent_errors.predicted].head(15))

Accuracy: 0.9994 Macro-F1: 0.9994
                            precision    recall  f1-score   support

        account_management     1.0000    1.0000    1.0000      1282
       billing_and_refunds     1.0000    1.0000    1.0000      1287
                 complaint     1.0000    1.0000    1.0000       798
greeting_goodbye_gratitude     1.0000    1.0000    1.0000        15
          order_management     0.9972    1.0000    0.9986      1053
              order_status     1.0000    0.9939    0.9969       491
              out_of_scope     1.0000    1.0000    1.0000        14

                  accuracy                         0.9994      4940
                 macro avg     0.9996    0.9991    0.9994      4940
              weighted avg     0.9994    0.9994    0.9994      4940



,text,gold,predicted
23573,i have got to locte order {{Order Number}},order_status,order_management
23196,purchase {{Order Number}} ststus,order_status,order_management
23773,need to trafk order {{Order Number}} help me,order_status,order_management


In [11]:
joblib.dump(intent_model,ARTIFACTS/'intent'/'intent_pipeline.joblib',compress=3)
intent_meta={
    'dataset':'bitext/Bitext-customer-support-llm-chatbot-training-dataset',
    'route_map':ROUTE_MAP,'route_labels':sorted(intent_df['route_intent'].unique()),
    'test_accuracy':float(intent_acc),'test_macro_f1':float(intent_f1),
    'augmentation':'Small-talk and out-of-scope examples added because Bitext lacks these labels.',
    'quick_mode':QUICK_MODE,'seed':SEED
}
(ARTIFACTS/'intent'/'metadata.json').write_text(json.dumps(intent_meta,indent=2),encoding='utf-8')
examples=['Hello!','Where is my order?','I need a refund','This service is terrible','Tell me a joke']
print(list(zip(examples,intent_model.predict(examples))))
assert intent_model.predict(['Where is my order?'])[0]=='order_status'
print('INTENT MODULE PASSED')

[('Hello!', 'greeting_goodbye_gratitude'), ('Where is my order?', 'order_status'), ('I need a refund', 'billing_and_refunds'), ('This service is terrible', 'complaint'), ('Tell me a joke', 'out_of_scope')]
INTENT MODULE PASSED


# Module 4 — Retrieval-Augmented Generation

The customer instruction is the retrieval key; its paired approved response becomes grounding context. `all-MiniLM-L6-v2` converts text into dense vectors. Vectors are normalized, and FAISS inner-product search therefore ranks by cosine similarity.

For a less misleading evaluation, the notebook first builds a temporary index from a stratified training partition and evaluates intent Recall@3 on held-out questions. It then builds the final deployment index from the complete knowledge base.

In [12]:
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

embedder = SentenceTransformer(
    EMBED_MODEL,
    device=str(DEVICE)
)

kb = bitext[
    ['instruction', 'response', 'category', 'intent']
].drop_duplicates(
    subset=['instruction']
).reset_index(drop=True)


retr_train, retr_test = train_test_split(
    kb,
    test_size=0.10,
    random_state=SEED,
    stratify=kb['intent']
)

if QUICK_MODE:
    retr_train = retr_train.groupby(
        'intent',
        group_keys=False
    ).head(250)

    retr_test = retr_test.groupby(
        'intent',
        group_keys=False
    ).head(25)


train_vectors = embedder.encode(
    retr_train['instruction'].tolist(),
    batch_size=256 if DEVICE.type == 'cuda' else 64,
    show_progress_bar=False,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype('float32')


eval_index = faiss.IndexFlatIP(train_vectors.shape[1])
eval_index.add(train_vectors)


query_vectors = embedder.encode(
    retr_test['instruction'].tolist(),
    batch_size=256 if DEVICE.type == 'cuda' else 64,
    show_progress_bar=False,
    normalize_embeddings=True,
    convert_to_numpy=True
).astype('float32')


_, eval_ids = eval_index.search(query_vectors, 3)


hits = []

train_intents = retr_train['intent'].reset_index(drop=True)

for gold_intent, retrieved_ids in zip(
    retr_test['intent'].tolist(),
    eval_ids
):
    is_hit = any(
        train_intents.iloc[int(retrieved_id)] == gold_intent
        for retrieved_id in retrieved_ids
    )
    hits.append(is_hit)

retrieval_recall3 = float(np.mean(hits))

print(
    f'Held-out intent Recall@3: '
    f'{retrieval_recall3:.4f} '
    f'({retrieval_recall3 * 100:.2f}%)'
)

del train_vectors, query_vectors, eval_index
gc.collect()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Held-out intent Recall@3: 0.9971 (99.71%)


171

In [13]:
# Final deployment index uses every available support pair.
kb_vectors=embedder.encode(kb['instruction'].tolist(),batch_size=256 if DEVICE.type=='cuda' else 64,
    show_progress_bar=True,normalize_embeddings=True,convert_to_numpy=True).astype('float32')
rag_index=faiss.IndexFlatIP(kb_vectors.shape[1]); rag_index.add(kb_vectors)
faiss.write_index(rag_index,str(ARTIFACTS/'rag'/'faiss.index'))
kb.to_parquet(ARTIFACTS/'rag'/'knowledge_base.parquet',index=False)
rag_meta={
    'embedding_model':EMBED_MODEL,'rows':len(kb),'dimension':int(kb_vectors.shape[1]),
    'index':'FAISS IndexFlatIP','similarity':'cosine via normalized inner product',
    'held_out_intent_recall_at_3':retrieval_recall3,'seed':SEED
}
(ARTIFACTS/'rag'/'metadata.json').write_text(json.dumps(rag_meta,indent=2),encoding='utf-8')

def retrieve(query,top_k=3):
    vector=embedder.encode([query],normalize_embeddings=True,convert_to_numpy=True).astype('float32')
    scores,ids=rag_index.search(vector,top_k)
    result=[]
    for score,idx in zip(scores[0],ids[0]):
        row=kb.iloc[int(idx)].to_dict(); row['score']=float(score); result.append(row)
    return result

display(pd.DataFrame(retrieve('How can I track my package?',5)))
assert rag_index.ntotal==len(kb)
print('RETRIEVAL MODULE PASSED')

Batches:   0%|          | 0/96 [00:00<?, ?it/s]

,instruction,response,category,intent,score
0,where do I check when my package is going to a...,We understand your need for information on tra...,DELIVERY,delivery_period,0.691739
1,where can I check when my package is going to ...,We completely understand your eagerness to tra...,DELIVERY,delivery_period,0.686025
2,how to see when my package is going to arrive?,We understand your curiosity about the deliver...,DELIVERY,delivery_period,0.675862
3,where can I see when my package is going to ar...,We understand your eagerness to track the prog...,DELIVERY,delivery_period,0.667432
4,I want assitsance to see when my package is go...,I'm glad you contacted us to us for assistance...,DELIVERY,delivery_period,0.663673


RETRIEVAL MODULE PASSED


## Secure Groq connection

The API key is retrieved on demand from Kaggle Secrets and is never printed or stored in exported artifacts. The generation model ID is configurable, allowing replacement if model availability changes.

In [15]:
def get_groq_key():
    key=os.getenv('GROQ_API_KEY')
    if key: return key
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret('GROQ_API_KEY')
    except Exception as exc:
        raise RuntimeError('Create and enable the Kaggle secret GROQ_API_KEY, then rerun this cell.') from exc

GROQ_MODEL=os.getenv('GROQ_MODEL','openai/gpt-oss-20b')
groq_client=Groq(api_key=get_groq_key())
print('Groq configured with model:',GROQ_MODEL)

Groq configured with model: openai/gpt-oss-20b


# Complete chatbot pipeline

Processing order matters:

1. Detect the original language.
2. Translate non-English input to English because sentiment, intent, and the knowledge base are English.
3. Predict tone and route intent.
4. Bypass retrieval for small talk and out-of-scope requests.
5. For support routes, retrieve top-k context and reject context below a similarity threshold.
6. Mark negative messages and complaints as priority; complaints also offer human escalation.
7. Generate in the original language using only retrieved responses.

The system has no live order database. It must never claim a real shipment status, refund status, date, policy, or completed action.

In [16]:
@torch.no_grad()
def classify_tone(text):
    batch=sentiment_tokenizer(text,return_tensors='pt',truncation=True,max_length=MAX_LENGTH).to(DEVICE)
    probs=torch.softmax(sentiment_model(**batch).logits,1)[0].cpu().numpy()
    idx=int(probs.argmax())
    label=sentiment_model.config.id2label.get(idx,sentiment_model.config.id2label.get(str(idx),str(idx)))
    return str(label).lower(),float(probs[idx])

def call_llm(messages,temperature=0.1,max_tokens=450):
    response=groq_client.chat.completions.create(
        model=GROQ_MODEL,messages=messages,temperature=temperature,max_tokens=max_tokens
    )
    return response.choices[0].message.content.strip()

def translate_to_english(text,language):
    if language=='en': return text
    return call_llm([
        {'role':'system','content':'Translate the message into concise English. Preserve IDs, numbers, names, and meaning. Return only the translation.'},
        {'role':'user','content':text}],temperature=0.0,max_tokens=200)

def direct_response(text,language,intent):
    instruction=(f'You are a concise e-commerce support assistant. Reply in language code {language}. '
                 f'The route is {intent}. If it is out of scope, state that you only handle store-support topics.')
    return call_llm([{'role':'system','content':instruction},{'role':'user','content':text}],temperature=0.2,max_tokens=180)

def answer_customer(message,top_k=3,threshold=0.22):
    message=normalize_text(message)
    if not message: raise ValueError('Message cannot be empty.')
    language=str(language_model.predict([message])[0])
    english_query=translate_to_english(message,language)
    sentiment,sentiment_confidence=classify_tone(english_query)
    intent=str(intent_model.predict([english_query])[0])
    priority=sentiment=='negative' or intent=='complaint'

    if intent in {'greeting_goodbye_gratitude','out_of_scope'}:
        answer=direct_response(message,language,intent)
        return {'answer':answer,'language':language,'english_query':english_query,
                'sentiment':sentiment,'sentiment_confidence':sentiment_confidence,
                'intent':intent,'priority':priority,'escalate':False,'sources':[]}

    candidates=retrieve(english_query,top_k)
    relevant=[item for item in candidates if item['score']>=threshold]
    if not relevant:
        fallback='I do not have reliable support information for this request. Please offer a human agent.'
        answer=direct_response(fallback,language,'out_of_scope')
        return {'answer':answer,'language':language,'english_query':english_query,
                'sentiment':sentiment,'sentiment_confidence':sentiment_confidence,
                'intent':intent,'priority':priority,'escalate':True,'sources':candidates}

    context='\n\n'.join(
        f"Source {i+1}; intent={item['intent']}; similarity={item['score']:.3f}\n"
        f"Past question: {item['instruction']}\nApproved response: {item['response']}"
        for i,item in enumerate(relevant)
    )
    system=(f'You are a professional e-commerce customer-support assistant. Reply in language code {language}. '
            'Use ONLY the retrieved support context. Never invent an order/refund status, date, link, policy, or action. '
            'If context is insufficient, say so and offer a human agent. '
            f'Detected sentiment={sentiment}; route={intent}; priority={priority}. '
            'If priority is true, begin with brief empathy. If the route is complaint, offer human escalation. '
            'Keep the response concise and practical.')
    user_prompt='Retrieved context: ' + context + ' Customer message: ' + message
    answer=call_llm([{'role':'system','content':system},{'role':'user','content':user_prompt}])
    safe_sources=[{'instruction':x['instruction'],'intent':x['intent'],'category':x['category'],
                   'score':round(x['score'],4)} for x in relevant]
    return {'answer':answer,'language':language,'english_query':english_query,
            'sentiment':sentiment,'sentiment_confidence':sentiment_confidence,
            'intent':intent,'priority':priority,'escalate':intent=='complaint','sources':safe_sources}

## End-to-end integration tests

The exact LLM wording is non-deterministic, so assertions test stable routing properties rather than matching a fixed sentence. Inspect each retrieved source to ensure it is relevant and verify that non-English responses use the original language.

In [17]:
test_messages=[
    'How can I track my order?',
    'I am furious! My refund still has not arrived.',
    'Your service is terrible and I need a human agent.',
    'Thank you for your help!',
    'Tell me the football score.',
    'أريد تغيير عنوان الشحن للطلب'
]
results=[]
for message in test_messages:
    print('\nUSER:',message)
    result=answer_customer(message)
    results.append(result)
    print('META:',{k:result[k] for k in ['language','sentiment','intent','priority','escalate']})
    print('BOT:',result['answer'])
    print('SOURCES:',result['sources'])

assert results[0]['intent']=='order_status'
assert results[1]['priority'] is True
assert results[3]['intent']=='greeting_goodbye_gratitude'
assert results[4]['intent']=='out_of_scope'
assert results[5]['language']=='ar'
print('END-TO-END INTEGRATION TESTS PASSED')


USER: How can I track my order?
META: {'language': 'en', 'sentiment': 'negative', 'intent': 'order_status', 'priority': True, 'escalate': False}
BOT: I’m sorry for any inconvenience. To help you track your order, could you please provide the order number?
SOURCES: [{'instruction': 'track order', 'intent': 'track_order', 'category': 'ORDER', 'score': 0.8157}, {'instruction': 'I want to track purchase {{Order Number}}, I need help', 'intent': 'track_order', 'category': 'ORDER', 'score': 0.8109}, {'instruction': 'i have got to track purchase {{Order Number}} help me', 'intent': 'track_order', 'category': 'ORDER', 'score': 0.8082}]

USER: I am furious! My refund still has not arrived.
META: {'language': 'en', 'sentiment': 'negative', 'intent': 'billing_and_refunds', 'priority': True, 'escalate': False}
BOT: I’m really sorry to hear how frustrating this has been for you.  
Could you please share your refund reference number or the order number associated with the refund? That will let me l

# Export everything

The archive contains all three trained classifiers, the FAISS index, knowledge-base metadata, and an evaluation summary. Download it after saving the Kaggle version. Keep it out of normal Git history because transformer weights and model artifacts can be large.

In [19]:
summary={
 'language':lang_meta,
 'sentiment':sentiment_meta,
 'intent':intent_meta,
 'rag':rag_meta,
 'groq_model':GROQ_MODEL,
 'pipeline_limitations':[
     'No connection to a real order or refund database',
     'Bitext support data is synthetic',
     'Emotion data is Twitter-domain English',
     'Non-English processing depends on translation',
     'Retrieval threshold requires validation on real user queries'
 ]
}
(ARTIFACTS/'evaluation_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
archive=shutil.make_archive(str(WORK/'complete_rag_artifacts'),'zip',ARTIFACTS)
print('Created:',archive)
print('Size MB:',round(Path(archive).stat().st_size/1024**2,2))
print('Files:')
for path in sorted(ARTIFACTS.rglob('*')):
    if path.is_file(): print(' -',path.relative_to(ARTIFACTS))

Created: /kaggle/working/complete_rag_artifacts.zip
Size MB: 293.47
Files:
 - evaluation_summary.json
 - intent/intent_pipeline.joblib
 - intent/metadata.json
 - language/language_pipeline.joblib
 - language/metadata.json
 - rag/faiss.index
 - rag/knowledge_base.parquet
 - rag/metadata.json
 - sentiment/config.json
 - sentiment/model.safetensors
 - sentiment/tokenizer.json
 - sentiment/tokenizer_config.json
 - sentiment/training_metadata.json


# What to discuss in the assessment

- **Language model:** character features are efficient and capture script and spelling patterns.
- **Sentiment model:** DistilBERT captures context, while weighted loss handles the small neutral bucket.
- **Intent model:** gold Bitext intents support supervised learning; word and character features handle phrases and typos.
- **RAG:** normalized dense embeddings plus FAISS provide semantic retrieval; the LLM receives only retrieved approved responses.
- **Routing:** greetings/out-of-scope messages bypass RAG; complaints and negative messages receive priority treatment.
- **Safety:** insufficient similarity triggers escalation, and the prompt forbids fabricated order/refund status.
- **Evaluation:** report actual metrics from this full run, not quick-mode results. Include language accuracy, sentiment macro-F1, intent macro-F1, held-out Recall@3, qualitative groundedness, and failure examples.
- **Limitations:** synthetic support data, sentiment domain shift, an imperfect neutral mapping, API dependence, and no transactional backend.